# Silver data cleaning: open charge FI

In [34]:
import pandas as pd
from pathlib import Path
import openpyxl

print(f"openpyxl version: {openpyxl.__version__}")
print(f"Current directory: {Path.cwd()}")

openpyxl version: 3.1.5
Current directory: c:\DE_Course\de-week8-mini-project-team1\Silver


## Bronze -> Silver transformation

In [6]:
# Read the Bronze-level data, the CSV file into a DataFrame
bronze_df = pd.read_csv("../Bronze/open_charge_raw_FI.csv", encoding="utf-8")

# Inspect the DataFrame
print(bronze_df.shape, "\n")
print(bronze_df.columns, "\n")
print(bronze_df.dtypes)

(4966, 86) 

Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
     

In [10]:
# Define the list of columns to keep and mapping, because we want to rename some columns
column_mapping = {
    "ID": "id",
    "NumberOfPoints": "number_of_points",
    "DateCreated": "date_created", # this one needs to be split later, only one key can exist in a Python dict
    "StatusType.IsOperational": "is_operational",
    "AddressInfo.Town": "municipality",
    "AddressInfo.StateOrProvince": "region",
    "AddressInfo.Country.Title": "country",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "fetch_timestamp": "fetch_timestamp"
}

# Select and rename
silver_df = (
    bronze_df[list(column_mapping.keys())]
    .rename(columns=column_mapping)
)

# Convert types
silver_df["date_created"] = pd.to_datetime(silver_df["date_created"])

# Derive columns
silver_df = silver_df.assign(
    year=silver_df["date_created"].dt.year,
    month=silver_df["date_created"].dt.month,
)

# Define the final column order explicitly
silver_df = silver_df[
    [
        "id",
        "number_of_points",
        "year",
        "month",
        "date_created",
        "is_operational",
        "municipality",
        "region",
        "country",
        "latitude",
        "longitude",
        "fetch_timestamp",
    ]
]

# Inspect the silver DataFrame
print(silver_df.shape, "\n")
print(silver_df.columns, "\n")
print(silver_df.dtypes)

(4966, 12) 

Index(['id', 'number_of_points', 'year', 'month', 'date_created',
       'is_operational', 'municipality', 'region', 'country', 'latitude',
       'longitude', 'fetch_timestamp'],
      dtype='str') 

id                                int64
number_of_points                float64
year                              int32
month                             int32
date_created        datetime64[us, UTC]
is_operational                   object
municipality                        str
region                              str
country                             str
latitude                        float64
longitude                       float64
fetch_timestamp                     str
dtype: object


In [19]:
# Convert the data types to use the nullable Pandas dtypes (Int64, boolean, string) for Silver-layer data
# because they preserve missing values (<NA>)

silver_df["number_of_points"] = silver_df["number_of_points"].astype("Int64")

silver_df["year"] = silver_df["year"].astype("Int64")
silver_df["month"] = silver_df["month"].astype("Int64")

# print(silver_df["is_operational"].unique()) # [True, <NA>, False]
silver_df["is_operational"] = silver_df["is_operational"].astype("boolean")

# use string of Pandas instead of str
string_columns = ["municipality", "region", "country", "fetch_timestamp"]
silver_df[string_columns] = silver_df[string_columns].astype("string")

# use datetime for the timestamp of fetching
silver_df["fetch_timestamp"] = pd.to_datetime(silver_df["fetch_timestamp"], errors="coerce", utc=True)

# Summary of data types:
# Int64 for nullable integers
# boolean for nullable booleans
# string for text
# float64 for coordinates
# datetime for date_created and fetch_timestamp

print(silver_df.dtypes)

id                                int64
number_of_points                  Int64
year                              Int64
month                             Int64
date_created        datetime64[us, UTC]
is_operational                  boolean
municipality                     string
region                           string
country                          string
latitude                        float64
longitude                       float64
fetch_timestamp     datetime64[us, UTC]
dtype: object


## Silver quality checks

In [ ]:
# Quality checks

# An approximate bounding box for mainland Finland (including Åland and Lapland)
# Latitude: 59.5° to 70.5°
# Longitude: 19.0° to 32.0°
FINLAND_LAT_MIN = 59.5
FINLAND_LAT_MAX = 70.5
FINLAND_LON_MIN = 19.0
FINLAND_LON_MAX = 32.0

quality_issues = []
total_rows = len(silver_df)

# Helper function for adding failed checks
def add_quality_issue(check, details, **extra_fields):
    quality_issues.append({
        "check": check,
        "status": "FAILED",
        "details": details,
        "total_rows": total_rows,
        **extra_fields
    })


# 1. Row count check
if len(silver_df) != len(bronze_df):
    add_quality_issue(
        check="ROW_COUNT",
        details=f"Bronze rows: {len(bronze_df)}, Silver rows: {len(silver_df)}",
        bronze_rows=len(bronze_df),
        silver_rows=len(silver_df)
    )


# 2. Duplicate check by id
duplicate_ids = silver_df[silver_df.duplicated(subset=["id"], keep=False)]

if not duplicate_ids.empty:
    add_quality_issue(
        check="DUPLICATE_IDS",
        details=f"Duplicate IDs found: {duplicate_ids['id'].nunique()}",
        affected_rows=len(duplicate_ids),
        duplicate_id_count=duplicate_ids["id"].nunique()
    )


# 3. Date range check - NOT USED, we want to keep all charging stations, even the ones added before 2016
# invalid_dates = silver_df[
#     silver_df["date_created"].dt.year.lt(2016)
#     | silver_df["date_created"].dt.year.gt(2026)
# ]

# if not invalid_dates.empty:
#     add_quality_issue(
#         check="DATE_RANGE",
#         details=f"Rows outside 2016-2026: {len(invalid_dates)}",
#         affected_rows=len(invalid_dates),
#         invalid_pct=round(len(invalid_dates) / total_rows * 100, 2)
#     )


# 4. Not-null checks
required_columns = [
    "id",
    "number_of_points",
    "date_created",
    "year",
    "month",
    "is_operational",
    "municipality",
    "country",
    "latitude",
    "longitude",
    "fetch_timestamp",
]

for col in required_columns:
    missing_count = silver_df[col].isna().sum()

    if missing_count > 0:
        add_quality_issue(
            check="NOT_NULL",
            details=f"{col}: {missing_count} missing values",
            column=col,
            missing_values=missing_count,
            non_null_values=silver_df[col].count(),
            missing_pct=round(missing_count / total_rows * 100, 2)
        )


# 5. Invalid numeric values: number_of_points
invalid_number_of_points = silver_df[
    silver_df["number_of_points"].lt(0)
]

if not invalid_number_of_points.empty:
    add_quality_issue(
        check="INVALID_COORDINATES",
        details=f"Invalid coordinate rows: {len(invalid_coordinates)}",
        affected_rows=len(invalid_coordinates),
        invalid_pct=round(len(invalid_coordinates) / total_rows * 100, 2)
    )


# 6. Invalid numeric values: coordinates
invalid_coordinates = silver_df[
    silver_df["latitude"].lt(FINLAND_LAT_MIN)
    | silver_df["latitude"].gt(FINLAND_LAT_MAX)
    | silver_df["longitude"].lt(FINLAND_LON_MIN)
    | silver_df["longitude"].gt(FINLAND_LON_MAX)
]

if not invalid_coordinates.empty:
    add_quality_issue(
        check="INVALID_COORDINATES",
        details=f"Invalid coordinate rows: {len(invalid_coordinates)}",
        affected_rows=len(invalid_coordinates)
    )

# 7. Invalid municipality

# Read the municipalities from the official Excel
# See https://stat.fi/en/luokitukset/tupa -> "Municipalities and Regional Divisions Based on Municipalities 2026"
municipalities_df = pd.read_excel(
    "en26_Municipalities_and_Regional_Divisions_Based_on_Municipalities_2026_in_Finnish,_Swedish_and_English.xlsx",
    usecols="A",
    skiprows=2,
    header=None,
    names=["municipality"] # add a header
)
# display(municipalities_df.head(10))
print(municipalities_df.shape)

valid_municipalities = set(
    municipalities_df["municipality"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.casefold()
)

invalid_municipalities = silver_df[
    silver_df["municipality"].notna()
    & ~silver_df["municipality"]
        .str.strip()
        .str.casefold()
        .isin(valid_municipalities)
].copy()

if not invalid_municipalities.empty:
    add_quality_issue(
        check="INVALID_MUNICIPALITY",
        details=f"Invalid municipality rows: {len(invalid_municipalities)}",
        affected_rows=len(invalid_municipalities),
        invalid_pct=round(len(invalid_municipalities) / total_rows * 100, 2)
    )


# 8. TODO: Invalid region



# Final quality report
quality_report = pd.DataFrame(quality_issues).fillna("") # replace NaN with blank cells so that it is more human-readable

if quality_report.empty:
    print("All quality checks passed.")
else:
    display(quality_report)

(308, 1)


,check,status,details,total_rows,column,missing_values,non_null_values,missing_pct,affected_rows,invalid_pct
0,NOT_NULL,FAILED,number_of_points: 3066 missing values,4966,number_of_points,3066.0,1900.0,61.74,,
1,NOT_NULL,FAILED,is_operational: 1280 missing values,4966,is_operational,1280.0,3686.0,25.78,,
2,NOT_NULL,FAILED,municipality: 12 missing values,4966,municipality,12.0,4954.0,0.24,,
3,INVALID_COORDINATES,FAILED,Invalid coordinate rows: 5,4966,,,,,5.0,
4,INVALID_MUNICIPALITY,FAILED,Invalid municipality rows: 1547,4966,,,,,1547.0,31.15


In [44]:
# Inspect the failed data in more detail

# invalid_municipalities
invalid_municipalities.head(20)

# invalid_dates
# display(invalid_coordinates)

,id,number_of_points,year,month,date_created,is_operational,municipality,region,country,latitude,longitude,fetch_timestamp
6,492112,2,2026,6,2026-06-01 18:35:00+00:00,True,Masala,<NA>,Finland,60.154667,24.529397,2026-06-30 07:14:05.354549+00:00
8,484655,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Nuorgam,<NA>,Finland,70.083900,27.909700,2026-06-30 07:14:05.354549+00:00
12,484651,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Karigasniemi,<NA>,Finland,69.467400,25.839100,2026-06-30 07:14:05.354549+00:00
13,484650,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Karigasniemi,<NA>,Finland,69.399362,25.851431,2026-06-30 07:14:05.354549+00:00
14,484649,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Kaamanen,<NA>,Finland,69.292837,26.724388,2026-06-30 07:14:05.354549+00:00
15,484648,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Kilpisjärvi,<NA>,Finland,69.043879,20.805370,2026-06-30 07:14:05.354549+00:00
16,484647,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Kilpisjärvi,<NA>,Finland,69.016800,20.870600,2026-06-30 07:14:05.354549+00:00
20,484643,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Ivalo,<NA>,Finland,68.736999,27.476900,2026-06-30 07:14:05.354549+00:00
21,484642,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Ivalo,<NA>,Finland,68.660103,27.542500,2026-06-30 07:14:05.354549+00:00
22,484641,<NA>,2026,4,2026-04-09 04:38:00+00:00,True,Ivalo,<NA>,Finland,68.659124,27.542145,2026-06-30 07:14:05.354549+00:00


In [ ]:
# TODO: drop the column "date_created", when it is not needed anymore for debugging purposes